Crawling vs invitational Stayman after `1N-2C-2D-2H`

This notebook compares two meanings for the responder's `2H` after `1N-2C-2D` when opener is 15-17 balanced and denies a 4-card major.

- `standard` meaning: `2H` is invitational, showing `5H-4S` with 8 HCP.
- `crawling` meaning: `2H` is weak, showing equal-length majors (4-4 weak).

The comparison is made for both responder hand types:
- 5H-4S invitational responder
- 4-4 weak responder

It estimates relative frequency and the expected score/IMP impact for each strategy under both vulnerability settings.

In [15]:
%use dataframe

@file:DependsOn("com.github.phisgr:rektdeal:0.3.0")

import com.github.phisgr.dds.*
import com.github.phisgr.rektdeal.*
import kotlin.math.*
import org.jetbrains.kotlinx.dataframe.api.*

In [16]:
@file:DependsOn("com.github.phisgr:rektdeal:0.3.0")
import com.github.phisgr.dds.*
import com.github.phisgr.rektdeal.*

fun contractScore(deal: com.github.phisgr.rektdeal.Deal, contract: com.github.phisgr.rektdeal.Contract, vulnerable: Boolean): Int {
    val tricks = deal.ddTricks(contract.strain, SOUTH)
    return contract.score(tricks, vulnerable)
}

fun hasFourCardMajor(north: com.github.phisgr.rektdeal.Hand): Boolean = north.hearts.size == 4 || north.spades.size == 4
fun isStrongOpener(north: com.github.phisgr.rektdeal.Hand): Boolean = north.hcp >= 16
fun weakInvOnTwoH(north: com.github.phisgr.rektdeal.Hand): Boolean = !isStrongOpener(north) && north.hearts.size == 3

fun invContract(deal: com.github.phisgr.rektdeal.Deal, useTwoH: Boolean): com.github.phisgr.rektdeal.Contract {
    val north = deal.north
    val has4H = north.hearts.size == 4
    val has4S = north.spades.size == 4

    if (has4H || has4S) {
        return if (isStrongOpener(north)) {
            if (has4H) Contract("4H") else Contract("4S")
        } else {
            if (has4H) Contract("3H") else Contract("3S")
        }
    }

    return when {
        weakInvOnTwoH(north) -> if (useTwoH) Contract("2H") else Contract("3H")
        !isStrongOpener(north) -> Contract("3H")
        north.hearts.size == 3 -> Contract("4H")
        else -> Contract("3N")
    }
}

fun standardContract(deal: com.github.phisgr.rektdeal.Deal, responderType: String): com.github.phisgr.rektdeal.Contract = when (responderType) {
    "4-4 weak" -> com.github.phisgr.rektdeal.Contract("1N")
    "5H4S inv" -> invContract(deal, useTwoH = true)
    else -> error("Unsupported responder type: $responderType")
}

fun crawlingContract(deal: com.github.phisgr.rektdeal.Deal, responderType: String): com.github.phisgr.rektdeal.Contract = when (responderType) {
    "4-4 weak" -> if (deal.south.hearts.size >= deal.south.spades.size) com.github.phisgr.rektdeal.Contract("2H") else com.github.phisgr.rektdeal.Contract("2S")
    "5H4S inv" -> invContract(deal, useTwoH = false)
    else -> error("Unsupported responder type: $responderType")
}

fun isFiveHeartFourSpadeInv(south: com.github.phisgr.rektdeal.Hand): Boolean =
    south.hearts.size == 5 && south.spades.size == 4 && south.hcp == 8

fun isFourFourWeak(south: com.github.phisgr.rektdeal.Hand): Boolean =
    south.hearts.size == 4 && south.spades.size == 4 && south.hcp <= 7

In [17]:
val northStack = SmartStack(Shape("(4333)") + Shape("(4432)") + Shape("(5332)"), Evaluator.hcp, 15..17)
val dealer = Dealer(N = northStack)

val sampleCount = 100

val payOffVul = PayOff(listOf("standard", "crawling"), PayOff.impFromScores)
val payOffNonVul = PayOff(listOf("standard", "crawling"), PayOff.impFromScores)

var totalDeals = 0
val responderCounts = mutableMapOf("5H4S inv" to 0, "4-4 weak" to 0)

multiThread(
    count = sampleCount,
    dealer = { Dealer(N = northStack) },
    accept = { deal ->
        isFiveHeartFourSpadeInv(deal.south) || isFourFourWeak(deal.south)
    }
) { dealCount, deal ->
    if (dealCount % 20 == 0) {
        println("Analyzing $dealCount deals...")
    }

    totalDeals++
    val south = deal.south
    val responderType = when {
        isFiveHeartFourSpadeInv(south) -> "5H4S inv"
        isFourFourWeak(south) -> "4-4 weak"
        else -> null
    }

    if (responderType == null) return@multiThread

    responderCounts[responderType] = responderCounts[responderType]!! + 1

    listOf(false to payOffNonVul, true to payOffVul).forEach { (isVul, payOff) ->
        val standardScore = contractScore(deal, standardContract(deal, responderType), isVul)
        val crawlingScore = contractScore(deal, crawlingContract(deal, responderType), isVul)

        payOff.addData(mapOf(
            "standard" to standardScore,
            "crawling" to crawlingScore,
        ))
    }
}

println("Sample complete.")
println("Total deals: $totalDeals")
println("Responder counts: $responderCounts")

println("Non-vulnerable IMP comparison:")
println(payOffNonVul)
println("Vulnerable IMP comparison:")
println(payOffVul)

Analyzing 20 deals...
Analyzing 40 deals...
Analyzing 60 deals...
Analyzing 80 deals...
Analyzing 100 deals...
Sample complete.
Total deals: 100
Responder counts: {5H4S inv=12, 4-4 weak=87}
Non-vulnerable IMP comparison:
        standar crawlin 
standar         +0.82   
                (0.33)  
crawlin -0.82           
        (0.33)          

Vulnerable IMP comparison:
        standar crawlin 
standar         +1.11   
                (0.44)  
crawlin -1.11           
        (0.44)          



In [18]:
payOffVul.toDataFrame()

strategy,standard,crawling
standard,null,+1.11 ± 0.44
crawling,-1.11 ± 0.44,null


### Interpretation notes

- The notebook uses `PayOff.impFromScores` to compare the average IMP impact of the two strategies directly from contract score outcomes.
- `standard` is the meaning where `2H` is invitational 5H-4S.
- `crawling` is the meaning where `2H` is weak equal-length majors and the 5H-4S invitational hand must be played using a higher-level contract.

The opener is modeled as 15-17 balanced with shapes `(4333)`, `(4432)`, and `(5332)` so both 3-card and 4-card major distributions are included.